# C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\DataSet Creation\01.5-DataSet-Auto-Labelling.ipynb

In [ ]:
def window_checking(array, window_len=0.2, min_num_picks=3):
    array.sort()
    s = np.diff(array).sum()
    # print(s)
    logging.info(f'{s}')
    if (s <= window_len) and (min_num_picks<=array.size):
        cond = True
    else:
        cond = False
    return cond

In [ ]:
def find_optimum_pick_time(times, outlier_detector='Z-score'):
    if outlier_detector=='Z-score':
        outlier_msk = srs.detect_outliers_ztest(array=times, threshold=1)
    elif outlier_detector=='IQR':
        outlier_msk = srs.detect_outliers_iqr(array=times, multiplier=1.5)
    times_inlier = times[~outlier_msk]
    # print(times_inlier)
    cond = window_checking(array=times_inlier,
                           window_len=1,
                           min_num_picks=2)
    # print(cond)
    logging.info(f'{cond}')
    if cond:
        time_optimum = srs.distance_weighted_average(array=times_inlier)
    else:
        time_optimum = np.nan
    # print(times_inlier)
    return time_optimum

In [ ]:
def get_picks_time_difference(picks):
    picks_time = [pick.time for pick in picks]
    picks_time = sorted(picks_time)
    picks_difftime = [time-picks_time[0] for time in picks_time]
    return picks_difftime

In [ ]:
def reversing_dictionary(dictionary):
    return {v:k for k, v in dictionary.items()}

In [ ]:
def find_peaks(data, treshold):
    mask = data > treshold
    labeled, num_features = label(mask)
    peaks = []
    for i in range(1, num_features + 1):
        segment_indices = np.where(labeled == i)[0]
        segment_values = data[segment_indices]
        max_index = np.argmax(segment_values)
        max_index_in_segment = segment_indices[np.argmax(segment_values)]
        peaks.append(max_index_in_segment)
    return peaks

# C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\Training\01.5-DataSet-QualityCheck.ipynb

In [ ]:
a = np.array([-1, 1, -2, 1, 1, -20, -1, 1, -1, 100])
# a.__abs__()
detector = srw.health_check.constant.DerivativeDetector(dt=0.01, spike_threshold=None)
result = detector.detect(a)
print(result)
# result.derivative
# diff =  np.diff(a)
# diff = np.pad(diff, (1, 0))
# diff / 0.01

# C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\Training\02.5-DataSet-Status.ipynb

In [ ]:
threshold_flatness = 0.01

keys = [key for key in df_std.keys()
        if key.endswith('_std')]
flat_signals = df_std[keys] <= threshold_flatness
numbers_of_flat_signals = flat_signals.sum(axis=1)
cond_not_flat = numbers_of_flat_signals <= 0

In [ ]:
treshold_snr = 2
keys = [key for key in df_metadata.keys() if key.endswith('_snr')]
cond_snr_channel = df_metadata[keys] >= treshold_snr
numbers_of_good_snr_channel = cond_snr_channel.sum(axis=1)
# numbers_of_good_snr_channel.hist()
cond_snr = numbers_of_good_snr_channel == 3

In [ ]:
func = lambda x: len(x) if isinstance(x, list) else 0
num_P_autopicks = df_autopicks[key_p].apply(func)
num_S_autopicks = df_autopicks[key_s].apply(func)

# cond_only_one_eq_in_window = (num_P_autopicks==1) & (num_S_autopicks==1)
cond_only_one_eq_in_window = num_S_autopicks <= 1

p_phase_time_difference = df_autopicks[key_p].apply(residual_pick_time)

excepted_error = 200                                                            # in samples
cond_if_P_phase_is_not_outlier = p_phase_time_difference <= excepted_error

In [ ]:
def residual_pick_time(auto_picks, manual_pick=500, unkown=9999):
    if isinstance(auto_picks, list) and (len(auto_picks)>0):
        auto_picks = np.array(auto_picks)
        rms = auto_picks - manual_pick
        rms = abs(rms)
        output = min(rms)
    else:
        output = unkown
    return output

In [ ]:
for channel in ['E', 'N', 'Z']:
    m_band = df_fft[f'trace_{channel}_max_M-band_fft']
    h_band = df_fft[f'trace_{channel}_max_H-band_fft']
    df_fft[f'trace_{channel}_noise_level'] = h_band / m_band

keys = [key for key in df_fft.keys() if key.endswith('_noise_level')]

treshold_noisy_level = 1
noisy_channel = df_fft[keys] >= treshold_noisy_level
numbers_of_noisy_channel = noisy_channel.sum(axis=1)
cond_not_noisy_data = numbers_of_noisy_channel == 0

# C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\Training\02.5-DataSet-std.ipynb

In [ ]:
std = data_X.std(axis=1)

# C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\Training\01.5-DataSet-QualityCheck.ipynb

In [ ]:
metadata = dataset.metadata.copy()
sps = dataset.sampling_rate
for ii in tqdm(metadata.index):
    data_3c, _ = dataset.get_sample(ii)
    for data, channel in zip(data_3c, dataset.component_order):
        freq, ampl = srw.waveform.fft(array=data, delta=1/sps)
        fft_low = ampl[freq<1]
        fft_mid = ampl[(1<=freq) & (freq<20)]
        fft_hig = ampl[20<=freq]
        metadata.at[ii, f'trace_{channel}_max_L-band_fft'] = fft_low.max().round(3)
        metadata.at[ii, f'trace_{channel}_max_M-band_fft'] = fft_mid.max().round(3)
        metadata.at[ii, f'trace_{channel}_max_H-band_fft'] = fft_hig.max().round(3)
    # break

# C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\Training\problematic data.ipynb

In [ ]:
def corr (a, b):
    """
    Row-wise (vectorized):
    """
    a_m = a - np.mean(a, axis=1, keepdims=True)
    b_m = b - np.mean(b, axis=1, keepdims=True)

    cov = np.sum(a_m * b_m, axis=1)
    std = np.std(a, axis=1) * np.std(b, axis=1) * a.shape[1]

    row_corr_vec = cov / std
    return row_corr_vec

In [ ]:
def visual_eval(data_X, data_Y, models, phases='NPS', title=None):
    results = {}
    for key, model in models.items():
        with torch.no_grad():
            data = torch.tensor(data_X, device=model.device).unsqueeze(0)
            pred = model(data)
            pred = pred[0].cpu().numpy()
        results[key] = pred
    #
    nrows = len(models) + 2
    gridspec_kw = {"hspace": 0,
                   "height_ratios": [3] + [1]*(nrows-1)}
    #
    fig, axes = plt.subplots(
        nrows, 1,
        figsize=(10, 6),
        sharex=True,
        gridspec_kw=gridspec_kw)
    axes[0].plot(data_X.T + np.array([0, 1, 2]))
    axes[0].set_ylabel('Waveform')
    axes[0].set_yticks([0, 0.5, 1])
    #
    axes[1].plot(data_Y.T)
    axes[1].set_ylabel('Manual')
    nrow = 2
    for key, val in results.items():
        axes[nrow].plot(val.T, label=[ch for ch in phases])
        axes[nrow].set_ylabel(key)
        axes[nrow].set_yticks([0.5, 1])
        axes[nrow].legend()
        cc = corr(a=val, b=data_Y)
        axes[nrow].set_title(str(cc))
        nrow += 1
    fig.suptitle(title)


In [ ]:
n = [18, 30,37,76,87,97,105,109,126,137,149,158,161,173,179,196,204,209, 217,
     222, 230,235,251,254,257,261,260,265,266,267,268,273,275,290,302,310,
     312,323,331,335,355,367,376,388,391,393]

In [ ]:
# Sample data (replace with your actual signal)
t = np.linspace(0, 1, 1000, False)  # 1 second
carrier_frequency = 50  # Hz
modulating_frequency = 5  # Hz
signal = np.cos(2 * np.pi * carrier_frequency * t) * (0.5 + 0.5 * np.cos(2 * np.pi * modulating_frequency * t))

# Calculate analytic signal
analytic_signal = scsig.hilbert(signal)

# Calculate envelope
envelope = np.abs(analytic_signal)

# Plot the results
plt.plot(t, signal, label='Original Signal')
plt.plot(t, envelope, label='Envelope', linewidth=2)
plt.plot(t, -envelope,  linewidth=2) # Plot negative envelope for visualization
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.title("Signal and its Envelope")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from scipy.signal import find_peaks

x = signal
data = np.sum(x**2, axis=0)
data = x ** 2
npts = x.size
samprate = 100
# Calculate analytic signal
box = np.ones(5)
data_smoothed = np.convolve(data, box)
peaks, _ = find_peaks(data_smoothed)

analytic_signal = scsig.hilbert(data)

# Calculate envelope
envelope = np.abs(analytic_signal)

plt.plot(data, label='Envelope', linewidth=2)

plt.plot(peaks, data[peaks], label='peaks', linewidth=1)
plt.legend()

In [ ]:
x = sample["X"]
for _x in x:
    freq, ampl = fft(array=_x, delta=0.01)
    plt.plot(freq, ampl)
plt.show()
data = np.sum(x**2, axis=0)
analytic_signal = scsig.hilbert(data)
# Calculate envelope
envelope = np.abs(analytic_signal)
cc = correlate(a=envelope, b=sample['y'][1:].sum(axis=0), shift=0, demean=True, normalize='naive', method='auto')

print(f'{index=} {cc=}')

In [ ]:
for _x, channel in zip(x, dataset.component_order):
    freq, ampl = fft(array=_x, delta=0.01)
    axes[0].plot(_x+jj, label=channel)
    axes[1].semilogx(freq, ampl, label=channel)
    ####
    fft_low = ampl[freq<1]
    fft_mid = ampl[(1<=freq) & (freq<20)]
    fft_hig = ampl[20<=freq]
    d = {f'trace_{channel}_fft_max_lt1hz': fft_low.max().round(3),
        f'trace_{channel}_fft_max_ge1hz_lt20hz': fft_mid.max().round(3),
        f'trace_{channel}_fft_max_gt20hz': fft_hig.max().round(3),
        ###
        f'trace_{channel}_fft_mean_lt1hz': fft_low.mean().round(3),
        f'trace_{channel}_fft_mean_ge1hz_lt20hz': fft_mid.mean().round(3),
        f'trace_{channel}_fft_mean_gt20hz': fft_hig.mean().round(3)}
    ######################################################################
    for mode in ['max', 'mean']:
        lst = [f'trace_{channel}_fft_{mode}_lt1hz',
                f'trace_{channel}_fft_{mode}_ge1hz_lt20hz',
                f'trace_{channel}_fft_{mode}_gt20hz']
        _result = [d[key] for key in lst]
        condition = {'calibration_signal': False,
                    'High frequency Noise': False}
        if not _result[0] <  _result[1]:
                condition['calibration_signal'] = True
        if  _result[1] <=  _result[2]*1.1:
            condition['High frequency Noise'] = True
        result.update(d)
        # plt.semilogx(freq, ampl)
        jj += 1
        print(channel, condition)

# C:\Users\ikahbasi\OneDrive\Applications\GitHub\SeisPhaseTune\Training\01.5-DataSet-Status.ipynb

In [ ]:
freq, ampl = srw.waveform.fft(array=data, delta=1/sps)
fft_low = ampl[freq<1]
fft_mid = ampl[(1<=freq) & (freq<20)]
fft_hig = ampl[20<=freq]
metadata.at[ii, f'trace_{channel}_fft_max_L-band'] = fft_low.max().round(3)
metadata.at[ii, f'trace_{channel}_fft_max_M-band'] = fft_mid.max().round(3)
metadata.at[ii, f'trace_{channel}_fft_max_H-band'] = fft_hig.max().round(3)
########################################################################
metadata.at[ii, f'trace_{channel}_skewness'] = skew(data)


channel = 'E'
noisy_e = metadata[f'trace_{channel}_fft_max_M-band'] < metadata[f'trace_{channel}_fft_max_H-band']
channel = 'N'
noisy_n = metadata[f'trace_{channel}_fft_max_M-band'] < metadata[f'trace_{channel}_fft_max_H-band']
channel = 'Z'
noisy_z = metadata[f'trace_{channel}_fft_max_M-band'] < metadata[f'trace_{channel}_fft_max_H-band']

outlier_noisy = (noisy_e & noisy_n & noisy_z)
sum(outlier_noisy)

In [ ]:
def get_phase_time(metadata):
    keys = list(filter(re.compile("trace_[PpSs].*_arrival_sample").match,
                       metadata.keys()))
    times = {key.lower(): val
             for key, val in metadata.items()
             if (key in keys) and not np.isnan(val)}
    p = None
    s = None
    for key, val in times.items():
        if key.startswith('trace_p'):
            p = val
        if key.startswith('trace_s'):
            s = val
    return p, s

In [ ]:
skewness_treshold = 3